# CanSat Data Analysis — GRASP & VAMOS Mission

**Mission date:** 5 February 2026 | **Launch site:** Dübendorf, Switzerland (448 m ASL)  
**Sensors:** GRASP (T, P, GPS alt., PM₂.₅/PM₁₀) · VAMOS science (CO₂, T, P) · VAMOS wind (speed, dir, IMU) · OBAMA external (T, P, RH)  
**External reference:** UWYO radiosonde, station 06610 (Payerne, CH), 2026-02-05 12:00 UTC

---

**How to run:** `Kernel → Restart & Run All`  
**Dependencies:** `numpy pandas matplotlib scipy openpyxl`  
(Install with `pip install numpy pandas matplotlib scipy openpyxl` if needed.)

---

## Analysis Pipeline

| # | Section | Goal |
|---|---------|------|
| 0 | Setup & Data Loading | Import libraries, download `utils.py` in Colab, load all datasets |
| 1 | Raw Data Overview | Inspect unprocessed sensor channels — no filtering or unit conversion |
| 2 | Data Quality Assessment | Identify and quantify four instrument artefacts |
| 3 | Making Data Comparable | Extract the descent window, harmonise units, build a common altitude axis |
| 4 | Pre-processing | Uniform resampling + zero-phase Butterworth low-pass filtering |
| 5 | Scientific Analysis | Lapse rate, PM₂.₅/PM₁₀, CO₂, wind spectra, UWYO radiosonde |
| 6 | Discussion & Conclusions | Synthesise findings, state limitations, draw physical conclusions |

## Research Question

> **How do temperature, pressure, particulate matter (PM₂.₅/PM₁₀), CO₂ and wind speed vary
> with altitude and time during the GRASP–VAMOS CanSat descent, and what do time-frequency
> methods reveal about lower-troposphere structure and sensor behaviour?**

### Scientific Motivation

The ISA standard atmosphere assumes a dry-adiabatic temperature lapse rate of **−6.5 K km⁻¹**
in the troposphere. Deviations indicate atmospheric stability anomalies — inversions in the
morning boundary layer, convective mixing on warm afternoons, or large-scale dynamical features.

Aerosol loading (PM₂.₅, PM₁₀) is highest within the **planetary boundary layer (PBL)**,
typically 0–2 km AGL, where turbulent mixing is weak and anthropogenic sources are concentrated.
Observing a vertical gradient from a CanSat provides a direct measurement of the aerosol profile
across this layer.

CO₂ in the Swiss Mittelland averages **420–430 ppm** under well-mixed conditions. Near-surface
traffic and domestic heating can push local concentrations above 1 000 ppm. The VAMOS NDIR
sensor captures this urban fingerprint over the Dübendorf launch site.

Time-frequency analysis (Welch PSD, STFT spectrogram) applied to pressure and acceleration
signals can detect **parachute pendulum oscillations**, broadband turbulence, and electronic noise
— information invisible in simple time-domain plots. Separating spectra by flight phase
(aircraft cruise, drop, ground) prevents the physically distinct regimes from being averaged
into a single, uninterpretable spectrum.

In [ ]:
# ── Section 0: Environment setup ─────────────────────────────────────────────
# Detect Google Colab and fetch utils.py from GitHub if the local file is absent.
import sys, importlib
from pathlib import Path

_IN_COLAB = 'google.colab' in sys.modules
if _IN_COLAB and not Path('utils.py').exists():
    import urllib.request
    _URL = ('https://raw.githubusercontent.com/demartinomattia'
            '/Space_Data_2/main/utils.py')
    urllib.request.urlretrieve(_URL, 'utils.py')
    print(f'Downloaded utils.py from {_URL}')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.fft import fft, fftfreq
from scipy.signal import spectrogram

import utils
from utils import (
    FIG_DIR, save_figure, plot_uwyo_skewt,
    # ISA atmosphere
    isa_alt, isa_press, isa_temp_c, baro_altitude_agl,
    # Signal processing
    resample_uniform, butter_lowpass, detrend_linear, compute_welch_psd,
    # Wind utilities
    met_to_uv,
    # Flight phase detection
    detect_vamos_drop,
    # Data loaders
    load_grasp, load_vamos_science, load_vamos_wind, load_obama, load_uwyo_sounding,
    # Summary helper
    print_dataset_summary,
    # Reference constants
    WHO_PM25_24H, WHO_PM10_24H, CO2_BACKGROUND_PPM, UWYO_SKEWT_URL,
)

plt.rcParams.update({
    'figure.dpi': 120, 'font.size': 10,
    'axes.grid': True, 'grid.alpha': 0.3,
    'axes.spines.top': False, 'axes.spines.right': False,
})

print(f'utils.py version: {utils.__version__}  |  figures → {FIG_DIR.resolve()}')

In [ ]:
# ── Data loading ─────────────────────────────────────────────────────────────
# All loaders live in utils.py; this cell is purely a call site.
# Row-0 artefact and CO₂ warm-up are NOT removed here — that happens after
# Section 2 (Data Quality), once we have characterised each artefact.

grasp_raw = load_grasp()
vamos     = load_vamos_science()
wind      = load_vamos_wind()
obama     = load_obama()
uwyo      = load_uwyo_sounding()

In [ ]:
# ── Post-load derivations ────────────────────────────────────────────────────
# 1. Clean GRASP: remove sensor-init row 0 (characterised in Section 2)
grasp_clean = grasp_raw.iloc[1:].copy()
grasp_clean['alt_baro'] = isa_alt(grasp_clean['press_Pa'].values)

# 2. VAMOS: exclude CO₂ warm-up zeros (NDIR needs ~7 min thermal equilibration)
vamos_co2 = vamos[vamos['co2_ppm'] > 0].copy()

# 3. VAMOS rows concurrent with the GRASP flight (shared absolute timestamps)
T0_g, T1_g = grasp_clean['t_ms'].iat[0], grasp_clean['t_ms'].iat[-1]
vamos_conc  = vamos[(vamos['t_ms'] >= T0_g) & (vamos['t_ms'] <= T1_g)].copy()

# 4. Automatically identify apogee and landing from the VAMOS pressure record
vamos_drop      = detect_vamos_drop(vamos)
vamos['alt_agl'] = vamos_drop['h_agl']

# 5. Assign a flight-phase label to every wind sample
wind['phase'] = 'cruise'
wind.loc[
    (wind['t_s'] >= vamos_drop['t_apogee_s']) &
    (wind['t_s'] <= vamos_drop['t_landing_s']),
    'phase'
] = 'drop'
wind.loc[wind['t_s'] > vamos_drop['t_landing_s'] + 30, 'phase'] = 'ground'

# 6. Effective sampling rates from median inter-sample interval
FS_G = 1000.0 / np.median(np.diff(grasp_clean['t_ms'].values))
FS_V = 1000.0 / np.median(np.diff(vamos['t_ms'].values))
FS_W = 1000.0 / np.median(np.diff(wind['t_ms'].values))

print_dataset_summary(grasp_clean, vamos, wind, obama, uwyo, vamos_drop, vamos_conc)

---
## Section 1 — Raw Data Overview

Before applying any filter, correction, or unit conversion, it is essential to inspect the data
exactly as it was recorded by each sensor. This step serves two purposes:

1. **Sanity check** — verify that sensors recorded physically plausible values and that CSV/Excel
   loading worked correctly.
2. **Artefact spotting** — obvious glitches (initialisation spikes, stuck flags, warm-up zeros)
   are most visible in the unprocessed signal.

> Nothing is filtered, resampled, or unit-converted in this section.
> Artefacts are left visible deliberately; they will be characterised in Section 2.

The four datasets are plotted on **separate time axes** because they powered on at different
moments: GRASP was active only during the brief descent (~176 s), while VAMOS science and
VAMOS wind ran continuously for ~100 min encompassing pre-launch, flight, and post-landing.

In [ ]:
# ── 1.1 GRASP — raw sensor channels ─────────────────────────────────────────
# All five channels plotted against relative time from first sample.
# The red dot marks row 0 — the sensor-initialisation artefact.

fig, axes = plt.subplots(5, 1, figsize=(12, 11), sharex=True)
fig.suptitle('GRASP — raw sensor channels (all rows, no preprocessing)',
             fontsize=12, fontweight='bold')

_channels = [
    ('temp_C',   'Temperature (°C)',   'tab:red'),
    ('press_Pa', 'Pressure (Pa)',      'tab:blue'),
    ('alt_m',    'GPS altitude (m)',   'tab:green'),
    ('pm25',     'PM₂.₅ (µg m⁻³)',   'tab:orange'),
    ('pm10',     'PM₁₀  (µg m⁻³)',   'tab:purple'),
]
for ax, (col, lbl, clr) in zip(axes, _channels):
    ax.plot(grasp_raw['t_rel'].values, grasp_raw[col].values,
            lw=0.7, color=clr, alpha=0.85)
    ax.scatter(grasp_raw['t_rel'].iat[0], grasp_raw[col].iat[0],
               color='red', s=70, zorder=5, label='Row-0 init artefact')
    ax.set_ylabel(lbl)
    ax.legend(fontsize=7, loc='upper right')

axes[-1].set_xlabel('Time from first sample (s)')
plt.tight_layout()
save_figure(fig, 'fig_raw_grasp.png')
plt.show()

print(f'GRASP: {len(grasp_raw)} rows | duration {grasp_raw["t_rel"].iat[-1]:.1f} s | fs ≈ {FS_G:.1f} Hz')
print(f'Row-0 pressure: {grasp_raw["press_Pa"].iat[0]:.0f} Pa  '
      f'vs row-1 onwards mean: {grasp_clean["press_Pa"].mean():.0f} Pa')

In [ ]:
# ── 1.2 VAMOS science — full recording vs flight window ──────────────────────
# VAMOS recorded for ~100 min, but the actual descent lasted only ~176 s.
# Left panels: full recording — the drop phase is barely visible.
# Right panels: zoomed to a ±10/15-min window around the drop, making it legible.

_h_agl    = vamos_drop['h_agl']
_airborne = np.where(_h_agl > 30)[0]
_t_win0 = max(vamos['t_s'].values[_airborne[0]]  - 600, vamos['t_s'].iat[0]) if len(_airborne) else vamos_drop['t_apogee_s'] - 600
_t_win1 = min(vamos['t_s'].values[_airborne[-1]] + 900, vamos['t_s'].iat[-1]) if len(_airborne) else vamos_drop['t_landing_s'] + 900

_vmask   = (vamos['t_s'] >= _t_win0) & (vamos['t_s'] <= _t_win1)
_vz      = vamos[_vmask].copy()
_t_rel_z = (_vz['t_s'].values - _t_win0) / 60
_t_apo_z = (vamos_drop['t_apogee_s']  - _t_win0) / 60
_t_lnd_z = (vamos_drop['t_landing_s'] - _t_win0) / 60

t_h_v    = vamos['t_rel'].values / 3600
_t_apo_h = (vamos_drop['t_apogee_s']  - vamos['t_s'].iat[0]) / 3600
_t_lnd_h = (vamos_drop['t_landing_s'] - vamos['t_s'].iat[0]) / 3600

_rows = [
    ('co2_ppm', 'CO₂ (ppm)',         'tab:brown'),
    ('temp_C',  'Temperature (°C)',   'tab:red'),
    ('alt_baro','Baro. alt. (m ASL)', 'tab:green'),
]
fig, axes = plt.subplots(3, 2, figsize=(14, 9), sharey='row')
fig.suptitle('VAMOS science — full recording (left) vs flight window (right)',
             fontsize=12, fontweight='bold')

for i, (col, lbl, clr) in enumerate(_rows):
    axes[i, 0].plot(t_h_v, vamos[col], lw=0.4, color=clr, alpha=0.8)
    axes[i, 0].axvspan(_t_apo_h, _t_lnd_h, color='tab:orange', alpha=0.35,
                       label='Drop phase' if i == 0 else None)
    axes[i, 0].set_ylabel(lbl)
    if i == 0:
        axes[i, 0].axhline(CO2_BACKGROUND_PPM, ls='--', color='gray', lw=1.2,
                           label=f'Background {CO2_BACKGROUND_PPM} ppm')
        axes[i, 0].legend(fontsize=7)
    if i == 2:
        axes[i, 0].set_xlabel('Time from VAMOS start (h)')

    axes[i, 1].plot(_t_rel_z, _vz[col], lw=0.9, color=clr, alpha=0.9)
    axes[i, 1].axvspan(_t_apo_z, _t_lnd_z, color='tab:orange', alpha=0.3)
    axes[i, 1].axvline(_t_apo_z, color='k', lw=0.8, ls='--',
                       label='Apogee'  if i == 0 else None)
    axes[i, 1].axvline(_t_lnd_z, color='k', lw=0.8, ls=':',
                       label='Landing' if i == 0 else None)
    if i == 0:
        axes[i, 1].axhline(CO2_BACKGROUND_PPM, ls='--', color='gray', lw=1.2)
        axes[i, 1].legend(fontsize=7)
    if i == 2:
        axes[i, 1].set_xlabel('Time in flight window (min)')

axes[0, 0].set_title('Full recording (~100 min) — drop barely visible')
axes[0, 1].set_title('Zoomed to useful interval')
plt.tight_layout()
save_figure(fig, 'fig_raw_vamos_science.png')
plt.show()

print(f'Useful window: {_t_win0/60:.0f}–{_t_win1/60:.0f} min from VAMOS start '
      f'({(_t_win1-_t_win0)/60:.0f} min = {(_t_win1-_t_win0)/vamos["t_rel"].iat[-1]*100:.0f}% of full recording)')

In [ ]:
# ── 1.3 VAMOS wind — full recording vs flight window ─────────────────────────
# Same layout as the science sensor: full recording (left) vs zoomed window (right).
# _t_win0 and _t_win1 are carried over from the VAMOS science cell above.

t_h_w     = wind['t_rel'].values / 3600
_t_apo_wh = (vamos_drop['t_apogee_s']  - wind['t_s'].iat[0]) / 3600
_t_lnd_wh = (vamos_drop['t_landing_s'] - wind['t_s'].iat[0]) / 3600
_t_apo_wz = (vamos_drop['t_apogee_s']  - _t_win0) / 60
_t_lnd_wz = (vamos_drop['t_landing_s'] - _t_win0) / 60

_wmask = (wind['t_s'] >= _t_win0) & (wind['t_s'] <= _t_win1)
_wz    = wind[_wmask].copy()
_t_wz  = (_wz['t_s'].values - _t_win0) / 60

_wind_rows = [
    ('wind_spd',     'Wind speed (m s⁻¹)',   'steelblue'),
    ('wind_dir',     'Wind dir (°)',          'tab:cyan'),
    ('wind_acc_vec', '|Acceleration| (m s⁻²)','tab:purple'),
]
fig, axes = plt.subplots(3, 2, figsize=(14, 8), sharey='row')
fig.suptitle('VAMOS wind — full recording (left) vs flight window (right)',
             fontsize=12, fontweight='bold')

for i, (col, lbl, clr) in enumerate(_wind_rows):
    axes[i, 0].plot(t_h_w, wind[col], lw=0.4, color=clr, alpha=0.8)
    axes[i, 0].axvspan(_t_apo_wh, _t_lnd_wh, color='tab:orange', alpha=0.3,
                       label='Drop phase' if i == 0 else None)
    axes[i, 0].set_ylabel(lbl)
    if i == 0:
        axes[i, 0].legend(fontsize=7)
    if i == 2:
        axes[i, 0].set_xlabel('Time from wind start (h)')

    axes[i, 1].plot(_t_wz, _wz[col], lw=0.9, color=clr, alpha=0.9)
    axes[i, 1].axvspan(_t_apo_wz, _t_lnd_wz, color='tab:orange', alpha=0.3)
    axes[i, 1].axvline(_t_apo_wz, color='k', lw=0.8, ls='--',
                       label='Apogee'  if i == 0 else None)
    axes[i, 1].axvline(_t_lnd_wz, color='k', lw=0.8, ls=':',
                       label='Landing' if i == 0 else None)
    if i == 0:
        axes[i, 1].legend(fontsize=7)
    if i == 2:
        axes[i, 1].set_xlabel('Time in flight window (min)')

axes[0, 0].set_title('Full recording (~100 min)')
axes[0, 1].set_title('Zoomed to useful interval')
plt.tight_layout()
save_figure(fig, 'fig_raw_vamos_wind.png')
plt.show()

In [ ]:
# ── 1.4 OBAMA external CanSat — raw data ────────────────────────────────────
# OBAMA carries a dual-sensor suite (two independent T/P/RH sensors).
# With only ~31 rows, each data point represents a meaningful measurement.

_ob_pairs = [
    ('first_temp_avg_C',    'second_temp_avg_C',    'Temperature (°C)'),
    ('first_pstat_avg_hPa', 'second_pstat_avg_hPa', 'Pressure (hPa)'),
    ('first_hum_avg_pct',   'second_hum_avg_pct',   'Relative humidity (%)'),
]
fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
fig.suptitle('OBAMA external CanSat — raw data (dual-sensor suite, ~31 rows)',
             fontsize=12, fontweight='bold')

for ax, (c1, c2, lbl) in zip(axes, _ob_pairs):
    d1 = obama[['Time_s', c1]].dropna()
    d2 = obama[['Time_s', c2]].dropna()
    ax.plot(d1['Time_s'], d1[c1], 'o-', ms=4, lw=1, color='tab:blue',   label='Sensor 1')
    ax.plot(d2['Time_s'], d2[c2], 's-', ms=4, lw=1, color='tab:orange', label='Sensor 2')
    ax.set_ylabel(lbl)
    ax.legend(fontsize=8)

axes[-1].set_xlabel('Time (s)')
plt.tight_layout()
save_figure(fig, 'fig_raw_obama.png')
plt.show()

print(f'OBAMA S1 T: {obama["first_temp_avg_C"].mean():.1f} °C  |  '
      f'S2 T: {obama["second_temp_avg_C"].mean():.1f} °C  |  '
      f'Inter-sensor bias: {abs(obama["first_temp_avg_C"].mean() - obama["second_temp_avg_C"].mean()):.1f} K')

---
## Section 2 — Data Quality Assessment

Four artefacts were identified during initial data inspection. Each is characterised below
before any analysis is performed, so that all subsequent results are based on clean data.

| # | Sensor | Issue | Physical cause | Action |
|---|--------|-------|---------------|--------|
| 1 | GRASP | **Row-0 pressure spike** (81 261 Pa ≡ ~1 820 m ASL vs ~87 000 Pa for all subsequent rows) | Microcontroller writes a default register value on first sample, before the sensor is initialised | **Exclude row 0** |
| 2 | VAMOS | **CO₂ = 0** for the first ~7 min | NDIR sensor requires warm-up to reach thermal equilibrium | **Exclude zero CO₂ samples** |
| 3 | VAMOS | **Tumbling flag stuck at 0** despite measured acceleration variance | On-board detection threshold set too conservatively | Note as limitation; use vector acceleration as surrogate |
| 4 | GRASP | **GPS altitude scatter** ±10–20 m around the barometric reference | Urban canyon multipath and receiver noise | Use barometric altitude as the primary vertical axis |

Understanding these artefacts **before** any analysis prevents misinterpreting genuine physical
signals as artefacts and vice versa.

In [ ]:
# ── 2.1 Error analysis — all four artefacts in one figure ────────────────────

fig = plt.figure(figsize=(14, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.48, wspace=0.42)
fig.suptitle('Data Quality Assessment — four identified artefacts',
             fontsize=12, fontweight='bold')

# (a) Row-0 pressure anomaly
ax = fig.add_subplot(gs[0, :2])
ax.plot(grasp_raw['t_rel'], grasp_raw['press_Pa'], 'o-', ms=3,
        color='tab:blue', alpha=0.6, label='All rows')
ax.scatter(grasp_raw['t_rel'].iat[0], grasp_raw['press_Pa'].iat[0],
           s=200, color='red', zorder=6,
           label=f'Row 0: {grasp_raw["press_Pa"].iat[0]:.0f} Pa  '
                 f'(≡ {isa_alt(grasp_raw["press_Pa"].iat[0]):.0f} m ASL)')
ax.axhline(grasp_clean['press_Pa'].mean(), ls='--', color='navy', lw=1.5,
           label=f'Rows 1-end mean: {grasp_clean["press_Pa"].mean():.0f} Pa')
ax.set_xlabel('Time from first sample (s)')
ax.set_ylabel('Pressure (Pa)')
ax.set_title('(a) GRASP row-0 initialisation anomaly')
ax.set_xlim(-0.05, grasp_raw['t_rel'].iat[1] * 5)
ax.legend(fontsize=8)

# (b) CO₂ warm-up
ax = fig.add_subplot(gs[0, 2])
t_min_v = vamos['t_rel'].values / 60
co2_all = vamos['co2_ppm'].values
valid   = co2_all > 0
warmup_min = t_min_v[valid][0]
ax.plot(t_min_v[valid], co2_all[valid], lw=0.7, color='tab:brown')
ax.axvspan(0, warmup_min, alpha=0.18, color='red',
           label=f'Warm-up: {warmup_min:.1f} min\n({(~valid).sum()} zero samples)')
ax.set_xlabel('Time (min)')
ax.set_ylabel('CO₂ (ppm)')
ax.set_title('(b) VAMOS CO₂ warm-up artefact')
ax.legend(fontsize=8)
ax.set_xlim(0, 30)

# (c) Tumbling flag
ax = fig.add_subplot(gs[1, :2])
ax.fill_between(wind['t_rel'] / 3600, wind['tumbling'],
                step='mid', color='crimson', alpha=0.7)
ax.set_ylim(-0.05, 1.3)
ax.set_yticks([0, 1])
ax.set_yticklabels(['Not tumbling', 'Tumbling'])
ax.set_xlabel('Time (h)')
ax.set_title(f'(c) Tumbling flag — {wind["tumbling"].mean()*100:.1f}% active '
             f'(threshold too conservative; use |accel| as surrogate)')

# (d) GPS vs barometric altitude residual
ax = fig.add_subplot(gs[1, 2])
resid = grasp_clean['alt_m'] - grasp_clean['alt_baro']
ax.hist(resid, bins=40, color='tab:green', alpha=0.75, edgecolor='none')
ax.axvline(resid.mean(), color='red',  lw=2,   label=f'Mean: {resid.mean():.1f} m')
ax.axvline(0,            color='gray', lw=1.2, ls='--', label='Zero bias')
ax.set_xlabel('GPS − barometric (m)')
ax.set_ylabel('Count')
ax.set_title(f'(d) GPS vs baro altitude residual  (σ = {resid.std():.1f} m)')
ax.legend(fontsize=8)

save_figure(fig, 'fig_error_analysis.png')
plt.show()

_jump_Pa = grasp_clean['press_Pa'].iat[0] - grasp_raw['press_Pa'].iat[0]
_jump_m  = isa_alt(grasp_clean['press_Pa'].iat[0]) - isa_alt(grasp_raw['press_Pa'].iat[0])
_dt_ms   = grasp_clean['t_ms'].iat[0] - grasp_raw['t_ms'].iat[0]
print(f'Row-0 jump: {_jump_Pa:.0f} Pa ≡ {_jump_m:.0f} m in {_dt_ms:.0f} ms '
      f'→ implied speed {_jump_m/_dt_ms*1000:.0f} m s⁻¹  (physically impossible)')
print(f'CO₂ warm-up: {warmup_min:.1f} min  |  {(~valid).sum()} excluded zero samples')
print(f'Tumbling flag: always 0 (0% active)')
print(f'GPS–baro residual: mean = {resid.mean():.1f} m,  σ = {resid.std():.1f} m')

---
## Section 3 — Making the Datasets Comparable

### The Problem

Direct comparison of the four datasets is not straightforward because:

| Issue | Detail |
|-------|--------|
| **Different durations** | GRASP: ~176 s descent · VAMOS/wind: ~100 min continuous · OBAMA: ~15 min |
| **Different time references** | Each sensor has its own millisecond counter from power-on; they are not synchronised |
| **Different pressure units** | GRASP reports Pa · VAMOS reports hPa · OBAMA reports hPa |
| **No shared altitude channel** | GPS altitude (GRASP) is noisy; no common barometric reference |

### The Strategy

| Step | Action | Rationale |
|------|--------|-----------|
| **A** | Convert all pressures to hPa | Harmonise units |
| **B** | Derive barometric altitude for every sensor using the ISA formula | Creates a consistent vertical axis |
| **C** | Detect the VAMOS descent phase automatically from the pressure profile | Isolates the physically relevant segment |
| **D** | Cross-validate using absolute timestamps to find VAMOS rows concurrent with GRASP | Independent check |
| **E** | Define a contextual *flight window* (apogee − 10 min to landing + 15 min) | Gives context around the drop |

### Barometric altitude — ISA formula

The ISA troposphere barometric formula converts measured pressure $P$ (Pa) to altitude $h$ (m ASL):

$$h = \frac{T_0}{\Lambda}\left[1 - \left(\frac{P}{P_0}\right)^{\!\frac{R_d \Lambda}{g}}\right]$$

with $T_0 = 288.15$ K, $\Lambda = 6.5 \times 10^{-3}$ K m⁻¹, $P_0 = 101\,325$ Pa,
$R_d = 287.058$ J kg⁻¹ K⁻¹, $g = 9.807$ m s⁻².  
This gives a self-consistent altitude estimate without requiring a co-located surface pressure measurement.

In [ ]:
# ── 3.1 Descent phase identification ─────────────────────────────────────────
# The detect_vamos_drop() function uses the pressure gradient to find the
# apogee (minimum pressure) and the landing (pressure returns to p₀).
# Here we visualise both the identification and the derived AGL altitude.

t_rel_v_min = vamos['t_rel'].values / 60
_apo_min    = (vamos_drop['t_apogee_s']  - vamos['t_s'].iat[0]) / 60
_lnd_min    = (vamos_drop['t_landing_s'] - vamos['t_s'].iat[0]) / 60

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
fig.suptitle('VAMOS — automated descent phase identification from pressure',
             fontweight='bold')

ax = axes[0]
ax.plot(t_rel_v_min, vamos['press_hPa'], lw=0.7, color='tab:blue', label='VAMOS pressure')
ax.axhline(vamos_drop['p0_hPa'], ls=':', color='0.4', lw=1.5,
           label=f'Ground ref. p₀ = {vamos_drop["p0_hPa"]:.1f} hPa')
ax.axvspan(_apo_min, _lnd_min, color='tab:orange', alpha=0.25, label='Identified drop phase')
ax.axvline(_apo_min, color='tab:orange', lw=1.2, ls='--')
ax.axvline(_lnd_min, color='tab:orange', lw=1.2, ls=':')
ax.set_ylabel('Pressure (hPa)')
ax.set_title('(a) Pressure time series — apogee at minimum pressure, landing when p → p₀')
ax.legend(fontsize=8)

ax = axes[1]
ax.plot(t_rel_v_min, vamos['alt_agl'], lw=0.7, color='tab:green', label='AGL altitude (baro)')
ax.axvspan(_apo_min, _lnd_min, color='tab:orange', alpha=0.25)
ax.axvline(_apo_min, color='tab:orange', lw=1.2, ls='--', label=f'Apogee: {vamos_drop["h_peak_m"]:.0f} m AGL')
ax.axvline(_lnd_min, color='tab:orange', lw=1.2, ls=':', label='Landing')
ax.set_ylabel('Altitude AGL (m)')
ax.set_xlabel('Time from VAMOS start (min)')
ax.set_title('(b) Barometric altitude AGL — derived from VAMOS pressure')
ax.legend(fontsize=8)

plt.tight_layout()
save_figure(fig, 'fig_descent_detection.png')
plt.show()

print(f'Apogee:  t = {vamos_drop["t_apogee_s"]:.0f} s  |  h = {vamos_drop["h_peak_m"]:.0f} m AGL')
print(f'Landing: t = {vamos_drop["t_landing_s"]:.0f} s')
print(f'Drop duration: {vamos_drop["drop_duration_s"]:.0f} s  |  '
      f'Mean descent rate: {vamos_drop["descent_rate_mps"]:.2f} m s⁻¹')

In [ ]:
# ── 3.2 Raw cross-dataset comparison: T and P vs barometric altitude ──────────
# All pressures converted to hPa; altitudes from ISA formula.
# No filtering — sensor noise is fully visible.
# This plot makes explicit where each dataset sits on the altitude axis.

_ob_m1 = obama['first_pstat_avg_hPa'].between(800, 1100)
_ob_m2 = obama['second_pstat_avg_hPa'].between(800, 1100)
_ob1_p = obama.loc[_ob_m1, 'first_pstat_avg_hPa'].values
_ob2_p = obama.loc[_ob_m2, 'second_pstat_avg_hPa'].values
_h_ob1 = isa_alt(_ob1_p * 100)
_h_ob2 = isa_alt(_ob2_p * 100)
_ob1_T = obama.loc[_ob_m1, 'first_temp_avg_C'].values
_ob2_T = obama.loc[_ob_m2, 'second_temp_avg_C'].values

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
fig.suptitle('Raw cross-dataset comparison — Temperature & Pressure vs barometric altitude\n'
             '(no filtering, common ISA altitude axis, GRASP P converted Pa → hPa)',
             fontweight='bold')

ax = axes[0]
ax.scatter(grasp_clean['temp_C'], grasp_clean['alt_baro'],
           s=3, alpha=0.25, color='tab:red',
           label=f'GRASP ({len(grasp_clean)} pts, raw ≈{FS_G:.0f} Hz)')
ax.scatter(vamos_conc['temp_C'], vamos_conc['alt_baro'],
           s=10, alpha=0.7, color='tab:blue',
           label=f'VAMOS concurrent ({len(vamos_conc)} pts)')
ax.scatter(_ob1_T, _h_ob1, s=55, color='tab:cyan',  zorder=5, label='OBAMA sensor 1')
ax.scatter(_ob2_T, _h_ob2, s=55, color='teal', marker='D', zorder=5, label='OBAMA sensor 2')
ax.set_xlabel('Temperature (°C)')
ax.set_ylabel('Barometric altitude (m ASL)')
ax.set_title('Temperature — raw, unfiltered')
ax.legend(fontsize=8)

ax = axes[1]
ax.scatter(grasp_clean['press_Pa'] / 100, grasp_clean['alt_baro'],
           s=3, alpha=0.25, color='tab:blue', label='GRASP (Pa → hPa, raw)')
ax.scatter(vamos_conc['press_hPa'], vamos_conc['alt_baro'],
           s=10, alpha=0.7, color='tab:green', label='VAMOS concurrent (hPa)')
ax.scatter(_ob1_p, _h_ob1, s=55, color='tab:cyan',  zorder=5, label='OBAMA sensor 1')
ax.scatter(_ob2_p, _h_ob2, s=55, color='teal', marker='D', zorder=5, label='OBAMA sensor 2')
ax.set_xlabel('Pressure (hPa)')
ax.set_ylabel('Barometric altitude (m ASL)')
ax.set_title('Pressure — raw, unfiltered')
ax.legend(fontsize=8)

plt.tight_layout()
save_figure(fig, 'fig_raw_comparison.png')
plt.show()

print('Observations from raw comparison:')
print(f'  GRASP T range at ~constant alt: {grasp_clean["temp_C"].min():.1f}–'
      f'{grasp_clean["temp_C"].max():.1f} °C  (noise spread visible)')
print(f'  VAMOS concurrent T: {vamos_conc["temp_C"].min():.1f}–{vamos_conc["temp_C"].max():.1f} °C')
print(f'  OBAMA S1/S2 mean T: {_ob1_T.mean():.1f} / {_ob2_T.mean():.1f} °C  (ground level)')
print(f'  GRASP mean baro alt: {grasp_clean["alt_baro"].mean():.0f} m  |  '
      f'VAMOS concurrent mean: {vamos_conc["alt_baro"].mean():.0f} m')

---
## Section 4 — Pre-processing

### Why pre-process?

GRASP samples at **≈38 Hz** with a non-uniform inter-sample interval (timestamp jitter from
the microcontroller scheduler). This creates two problems:

1. **Non-uniform grid** — FFT and Welch PSD require equally-spaced samples.
2. **High-frequency noise** — electronic noise contaminates the physical temperature and
   pressure signals, which vary slowly during a parachute descent.

### Processing pipeline (applied to GRASP only)

```
Raw GRASP signal (≈38 Hz, non-uniform timestamps)
  │
  ├── Step 1: Linear interpolation onto a uniform 38 Hz grid
  │           → removes timestamp jitter without altering the signal spectrum
  │
  └── Step 2: Zero-phase Butterworth low-pass filter (scipy.signal.filtfilt)
              • Pressure   → −3 dB cutoff at 0.5 Hz
              • Temperature → −3 dB cutoff at 0.3 Hz  (thermal response is slower)
              → attenuates electronic noise; preserves the descent profile
```

**Why zero-phase (forward-backward)?** A causal filter introduces a time shift proportional
to its order. `filtfilt` applies the filter twice (forward then backward), achieving zero
phase shift at the cost of twice the effective order, which is exactly what we want for
clean altitude–variable profiles.

**Cutoff choices:** physical pressure fluctuations during a slow parachute descent are
well below 0.5 Hz; temperature responds even more slowly (thermal inertia of the sensor
housing). Noise is broadband and extends well above 1 Hz in the raw spectrum.

VAMOS and OBAMA are **not** resampled or filtered — their lower native sample rates
(≈1–2 Hz) already place them below the noise regime that affects GRASP.

In [ ]:
# ── 4.1 GRASP: uniform resampling + Butterworth LP filter ────────────────────

t_rel_g   = grasp_clean['t_rel'].values
press_raw = grasp_clean['press_Pa'].values
temp_raw  = grasp_clean['temp_C'].values

t_uni,     press_uni  = resample_uniform(t_rel_g, press_raw, FS_G)
_t_uni_T,  temp_uni   = resample_uniform(t_rel_g, temp_raw,  FS_G)

press_filt     = butter_lowpass(press_uni, cutoff_hz=0.5, fs=FS_G)
temp_filt      = butter_lowpass(temp_uni,  cutoff_hz=0.3, fs=FS_G)
grasp_filt_alt = isa_alt(press_filt)

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
fig.suptitle('GRASP — effect of uniform resampling + Butterworth low-pass filter',
             fontweight='bold')

axes[0].plot(t_uni, press_uni / 100, lw=0.6, alpha=0.45, color='tab:blue',  label='Raw (resampled)')
axes[0].plot(t_uni, press_filt / 100, lw=1.8, color='navy',                  label='Butterworth LP 0.5 Hz')
axes[0].set_ylabel('Pressure (hPa)')
axes[0].set_title('Pressure — raw vs filtered')
axes[0].legend(fontsize=8)

axes[1].plot(t_uni, temp_uni,  lw=0.6, alpha=0.45, color='tab:red',  label='Raw (resampled)')
axes[1].plot(t_uni, temp_filt, lw=1.8, color='darkred',               label='Butterworth LP 0.3 Hz')
axes[1].set_ylabel('Temperature (°C)')
axes[1].set_xlabel('Time from ejection (s)')
axes[1].set_title('Temperature — raw vs filtered')
axes[1].legend(fontsize=8)

plt.tight_layout()
save_figure(fig, 'fig_filter_effect.png')
plt.show()

print(f'Uniform grid: {len(t_uni)} pts at fs ≈ {FS_G:.2f} Hz')
print(f'Pressure cutoff 0.5 Hz  |  Temperature cutoff 0.3 Hz  |  Filter order 4 (Butterworth, zero-phase)')

In [ ]:
# ── 4.2 Adjusted cross-dataset comparison: filtered GRASP + VAMOS + OBAMA + ISA
# After filtering, we can overlay all datasets and the ISA reference on the same
# altitude axis, making inter-sensor biases and the observed lapse rate visible.

_h_gnd = vamos_conc['alt_baro'].median() if len(vamos_conc) > 0 else isa_alt(vamos['press_hPa'].median() * 100)
_T_gnd = vamos_conc['temp_C'].median()   if len(vamos_conc) > 0 else vamos['temp_C'].median()

_dT   = _T_gnd - temp_filt.mean()
_dh   = _h_gnd - grasp_filt_alt.mean()
lapse_obs = _dT / (_dh / 1000)    # K km⁻¹  (positive = temperature increases with altitude = inversion)

h_isa = np.linspace(400, 2000, 400)
T_isa = isa_temp_c(h_isa)
P_isa = isa_press(h_isa)

mask1 = obama['first_temp_avg_C'].between(-20, 50)  & obama['first_pstat_avg_hPa'].notna()
mask2 = obama['second_temp_avg_C'].between(-20, 50) & obama['second_pstat_avg_hPa'].notna()
ob1   = obama[mask1]
ob2   = obama[mask2]
T1, P1 = ob1['first_temp_avg_C'].values,  ob1['first_pstat_avg_hPa'].values
T2, P2 = ob2['second_temp_avg_C'].values, ob2['second_pstat_avg_hPa'].values
h_ob1  = isa_alt(P1 * 100)
h_ob2  = isa_alt(P2 * 100)

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
fig.suptitle('Adjusted comparison — filtered GRASP + VAMOS + OBAMA + ISA reference',
             fontweight='bold')

ax = axes[0]
ax.scatter(temp_filt, grasp_filt_alt,
           s=4, alpha=0.5, color='tab:red', label='GRASP filtered (0.3 Hz LP)')
ax.scatter(_T_gnd, _h_gnd, s=140, color='tab:green', marker='^', zorder=6,
           label=f'VAMOS concurrent median ({_T_gnd:.1f} °C, {_h_gnd:.0f} m)')
ax.scatter(T1, h_ob1, s=45, color='tab:cyan',  zorder=5, label='OBAMA sensor 1')
ax.scatter(T2, h_ob2, s=45, color='teal', marker='D', zorder=5, label='OBAMA sensor 2')
ax.plot(T_isa, h_isa, 'k--', lw=2.0, label='ISA (−6.5 K km⁻¹)')
ax.plot([_T_gnd, temp_filt.mean()], [_h_gnd, grasp_filt_alt.mean()],
        'b-', lw=2.5, label=f'Observed lapse: {lapse_obs:.1f} K km⁻¹')
ax.set_xlabel('Temperature (°C)')
ax.set_ylabel('Altitude (m ASL)')
ax.set_title('Temperature–altitude profile')
ax.legend(fontsize=8)

ax = axes[1]
ax.scatter(press_filt / 100, grasp_filt_alt,
           s=4, alpha=0.5, color='tab:blue', label='GRASP filtered (0.5 Hz LP, hPa)')
ax.scatter(vamos_conc['press_hPa'].mean(), _h_gnd,
           s=140, color='tab:green', marker='^', zorder=6, label='VAMOS concurrent')
ax.scatter(P1, h_ob1, s=45, color='tab:cyan',  zorder=5, label='OBAMA sensor 1')
ax.scatter(P2, h_ob2, s=45, color='teal', marker='D', zorder=5, label='OBAMA sensor 2')
ax.plot(P_isa / 100, h_isa, 'k--', lw=2.0, label='ISA')
ax.set_xlabel('Pressure (hPa)')
ax.set_ylabel('Altitude (m ASL)')
ax.set_title('Pressure–altitude profile')
ax.legend(fontsize=8)

plt.tight_layout()
save_figure(fig, 'fig_adjusted_comparison.png')
plt.show()

T_isa_g = isa_temp_c(grasp_filt_alt.mean())
print(f'GRASP mean altitude (baro): {grasp_filt_alt.mean():.0f} m ASL')
print(f'ISA temperature at that altitude: {T_isa_g:.1f} °C  |  GRASP filtered mean: {temp_filt.mean():.1f} °C')
print(f'Warm bias vs ISA: +{temp_filt.mean() - T_isa_g:.1f} K')
print(f'Observed lapse rate (VAMOS–GRASP two-point): {lapse_obs:.1f} K km⁻¹  (ISA: −6.5 K km⁻¹)')
print(f'OBAMA S1 mean T: {T1.mean():.1f} °C  vs  VAMOS concurrent: {_T_gnd:.1f} °C  '
      f'(bias: {abs(T1.mean()-_T_gnd):.1f} K)')

---
## Section 5 — Scientific Analysis

With clean, comparable datasets in hand, we now address each component of the research question
in turn. Every subsection focuses on a specific physical quantity or sensor system.

### 5.1 Signal Processing — GRASP Pressure & Temperature

The detrended GRASP signals are analysed in the frequency domain using three complementary methods:

| Method | Description | Strength |
|--------|-------------|----------|
| **Hanning-windowed FFT** | Single-shot power spectrum from the full time series | High frequency resolution |
| **Welch PSD** | Averaged estimate from overlapping 256-sample segments (50% overlap) | Reduced variance |
| **STFT spectrogram** | Time-evolving spectrum, 128-sample window, 75% overlap | Reveals non-stationarities |

**Detrending** (linear subtraction) is applied before FFT/Welch to remove the dominant monotonic
descent trend, which otherwise produces spectral leakage that masks weaker periodic components.

**What to look for:**
- A sharp spectral peak would indicate a **parachute pendulum oscillation** (expected ~0.1–0.5 Hz).
- A broadband power-law $f^{-5/3}$ slope would indicate **Kolmogorov turbulence**.
- A flat white-noise floor at high frequency would indicate **electronic sensor noise**.
- Non-stationarity in the spectrogram would indicate **changing dynamics** (e.g., oscillation onset).

In [ ]:
# ── 5.1 GRASP signal processing: FFT / Welch PSD / STFT spectrogram ──────────
# Detrend linearly (removes the monotonic descent trend), then analyse
# the residual fluctuations in the frequency domain.

press_detr = detrend_linear(press_uni, t_uni)
temp_detr  = detrend_linear(temp_uni,  t_uni)

N    = len(press_detr)
hw   = np.hanning(N)
nrm  = np.sqrt(np.sum(hw ** 2))

psd_P_fft = (np.abs(fft(press_detr * hw))[:N//2] / nrm) ** 2 * (2 / N)
psd_T_fft = (np.abs(fft(temp_detr  * hw))[:N//2] / nrm) ** 2 * (2 / N)
freqs_fft = fftfreq(N, 1.0 / FS_G)[:N//2]

f_wP, Pxx_P = compute_welch_psd(t_uni, press_detr, fs=FS_G, nperseg=256)[:2]
f_wT, Pxx_T = compute_welch_psd(t_uni, temp_detr,  fs=FS_G, nperseg=256)[:2]

f_spec, t_spec, Sxx_P = spectrogram(
    press_detr, fs=FS_G, window='hann', nperseg=128, noverlap=96, nfft=256)
_, _, Sxx_T = spectrogram(
    temp_detr, fs=FS_G, window='hann', nperseg=128, noverlap=96, nfft=256)

fig = plt.figure(figsize=(14, 11))
gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.42, wspace=0.35)
fig.suptitle('GRASP — frequency-domain analysis (detrended pressure & temperature)',
             fontweight='bold')

ax = fig.add_subplot(gs[0, 0])
ax.semilogy(freqs_fft[1:], psd_P_fft[1:], lw=0.8, color='tab:blue', alpha=0.7, label='Hanning FFT')
ax.semilogy(f_wP[1:], Pxx_P[1:], lw=2.0, color='navy', label='Welch PSD')
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('PSD (Pa² Hz⁻¹)')
ax.set_title('(a) Pressure power spectrum')
ax.set_xlim(0, FS_G / 2)
ax.legend(fontsize=8)

ax = fig.add_subplot(gs[0, 1])
ax.semilogy(freqs_fft[1:], psd_T_fft[1:], lw=0.8, color='tab:red', alpha=0.7, label='Hanning FFT')
ax.semilogy(f_wT[1:], Pxx_T[1:], lw=2.0, color='darkred', label='Welch PSD')
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('PSD (°C² Hz⁻¹)')
ax.set_title('(b) Temperature power spectrum')
ax.set_xlim(0, FS_G / 2)
ax.legend(fontsize=8)

ax = fig.add_subplot(gs[1, 0])
ax.plot(t_uni, press_uni / 100, lw=0.6, alpha=0.45, color='tab:blue', label='Raw')
ax.plot(t_uni, press_filt / 100, lw=1.8, color='navy', label='Butterworth 0.5 Hz')
ax.set_ylabel('Pressure (hPa)')
ax.set_xlabel('Time (s)')
ax.set_title('(c) Pressure — raw vs filtered')
ax.legend(fontsize=8)

ax = fig.add_subplot(gs[1, 1])
ax.plot(t_uni, temp_uni, lw=0.6, alpha=0.45, color='tab:red', label='Raw')
ax.plot(t_uni, temp_filt, lw=1.8, color='darkred', label='Butterworth 0.3 Hz')
ax.set_ylabel('Temperature (°C)')
ax.set_xlabel('Time (s)')
ax.set_title('(d) Temperature — raw vs filtered')
ax.legend(fontsize=8)

ax = fig.add_subplot(gs[2, :])
pcm = ax.pcolormesh(t_spec, f_spec,
                    10 * np.log10(Sxx_P + 1e-12),
                    cmap='inferno', shading='gouraud', vmin=-30)
fig.colorbar(pcm, ax=ax, label='dB re Pa² Hz⁻¹')
ax.set_xlabel('Time from ejection (s)')
ax.set_ylabel('Frequency (Hz)')
ax.set_ylim(0, min(2.0, FS_G / 2))
ax.set_title(f'(e) STFT spectrogram — GRASP pressure  '
             f'(128-sample window ≈ 3.3 s at {FS_G:.0f} Hz, 75% overlap)')

save_figure(fig, 'fig_signal_processing.png')
plt.show()

print(f'Nyquist frequency (GRASP): {FS_G/2:.2f} Hz')
print(f'Welch frequency resolution Δf: {f_wP[1]:.4f} Hz')
print('No significant spectral peak above 0.1 Hz → no parachute pendulum resonance detected')

### 5.2 Particulate Matter — PM₂.₅ and PM₁₀

The GRASP optical particle counter records PM₂.₅ and PM₁₀ at ~38 Hz throughout the descent.
We compare the measured concentrations against the
**WHO 2021 Air Quality Guidelines** (24-h mean limits):

| Pollutant | WHO 2021 limit | Unit |
|-----------|---------------|------|
| PM₂.₅     | 15            | µg m⁻³ (24-h mean) |
| PM₁₀      | 45            | µg m⁻³ (24-h mean) |

Physically, we expect aerosol concentrations to **increase towards the surface** because:
- Anthropogenic sources (traffic, heating) are located near the ground.
- Turbulent vertical mixing in the planetary boundary layer is weaker near the bottom.
- Gravitational settling of coarser particles is most effective close to the surface.

The GRASP altitude range is narrow (~40 m barometrically), so any vertical gradient
is a lower bound on the true profile across the full PBL.

In [ ]:
# ── 5.2 Particulate matter analysis ──────────────────────────────────────────

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
fig.suptitle('GRASP — Particulate matter analysis and WHO 2021 comparison',
             fontweight='bold')

# (a) PM vs altitude
ax = axes[0, 0]
ax.scatter(grasp_clean['pm25'], grasp_clean['alt_baro'],
           s=5, alpha=0.6, color='tab:orange', label='PM₂.₅')
ax.scatter(grasp_clean['pm10'], grasp_clean['alt_baro'],
           s=5, alpha=0.4, color='tab:purple', label='PM₁₀')
ax.axvline(WHO_PM25_24H, color='tab:orange', ls='--', lw=1.5,
           label=f'WHO PM₂.₅ 24-h: {WHO_PM25_24H} µg m⁻³')
ax.axvline(WHO_PM10_24H, color='tab:purple', ls='--', lw=1.5,
           label=f'WHO PM₁₀ 24-h: {WHO_PM10_24H} µg m⁻³')
ax.set_xlabel('Concentration (µg m⁻³)')
ax.set_ylabel('Barometric altitude (m ASL)')
ax.set_title('(a) PM vs altitude')
ax.legend(fontsize=8)

# (b) PM vs time
ax = axes[0, 1]
ax.plot(grasp_clean['t_rel'], grasp_clean['pm25'], color='tab:orange', lw=1.0, label='PM₂.₅')
ax.plot(grasp_clean['t_rel'], grasp_clean['pm10'], color='tab:purple', lw=1.0, label='PM₁₀')
ax.axhline(WHO_PM25_24H, color='tab:orange', ls='--', lw=1.2, alpha=0.7)
ax.axhline(WHO_PM10_24H, color='tab:purple', ls='--', lw=1.2, alpha=0.7)
ax.set_xlabel('Time from ejection (s)')
ax.set_ylabel('Concentration (µg m⁻³)')
ax.set_title('(b) PM vs time')
ax.legend(fontsize=8)

# (c) PM₂.₅ distribution
ax = axes[1, 0]
vals25 = grasp_clean['pm25'].values
ax.hist(vals25, bins=30, color='tab:orange', alpha=0.75, edgecolor='none')
ax.axvline(WHO_PM25_24H,  color='red',   lw=2.0, ls='--',
           label=f'WHO 24-h limit: {WHO_PM25_24H} µg m⁻³')
ax.axvline(vals25.mean(), color='black', lw=1.8,
           label=f'Observed mean: {vals25.mean():.1f} µg m⁻³')
frac25 = (vals25 > WHO_PM25_24H).mean() * 100
ax.set_xlabel('PM₂.₅ (µg m⁻³)')
ax.set_ylabel('Sample count')
ax.set_title(f'(c) PM₂.₅ distribution  ({frac25:.0f}% above WHO limit)')
ax.legend(fontsize=8)

# (d) PM₁₀ distribution
ax = axes[1, 1]
vals10 = grasp_clean['pm10'].values
ax.hist(vals10, bins=30, color='tab:purple', alpha=0.75, edgecolor='none')
ax.axvline(WHO_PM10_24H,  color='red',   lw=2.0, ls='--',
           label=f'WHO 24-h limit: {WHO_PM10_24H} µg m⁻³')
ax.axvline(vals10.mean(), color='black', lw=1.8,
           label=f'Observed mean: {vals10.mean():.1f} µg m⁻³')
frac10 = (vals10 > WHO_PM10_24H).mean() * 100
ax.set_xlabel('PM₁₀ (µg m⁻³)')
ax.set_ylabel('Sample count')
ax.set_title(f'(d) PM₁₀ distribution  ({frac10:.0f}% above WHO limit)')
ax.legend(fontsize=8)

plt.tight_layout()
save_figure(fig, 'fig_pm_analysis.png')
plt.show()

print(f'PM₂.₅: mean = {vals25.mean():.1f}  max = {vals25.max():.0f} µg m⁻³  ({frac25:.0f}% > WHO 24-h limit)')
print(f'PM₁₀:  mean = {vals10.mean():.1f}  max = {vals10.max():.0f} µg m⁻³  ({frac10:.0f}% > WHO 24-h limit)')

### 5.3 CO₂ Concentration — VAMOS

VAMOS recorded CO₂ continuously for ~100 min using an NDIR (non-dispersive infrared) sensor.
The first ~7 min (warm-up, zero readings) are excluded from all CO₂ analysis.

Key reference: the **global atmospheric background** is approximately **420 ppm** as of 2024
(NOAA Mauna Loa observatory). Values above this indicate local sources — predominantly
road traffic and combustion in the peri-urban Dübendorf environment.

**Caveat:** NDIR sensors are temperature-sensitive. Without factory temperature-compensation
data, absolute accuracy is limited to roughly ±50 ppm. Relative trends within the dataset
are more reliable than absolute values.

In [ ]:
# ── 5.3 VAMOS science — CO₂, temperature and altitude in the flight window ───
# The orange band marks the drop phase (apogee → landing).
# All three panels share the same x-axis (minutes from window start).

_vz_co2  = _vz[_vz['co2_ppm'] > 0]
_t_co2_z = (_vz_co2['t_s'].values - _t_win0) / 60

fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
fig.suptitle(f'VAMOS science — flight window  '
             f'(t = {_t_win0/60:.0f}–{_t_win1/60:.0f} min from VAMOS start)',
             fontweight='bold')

axes[0].plot(_t_co2_z, _vz_co2['co2_ppm'], lw=0.6, color='tab:brown', alpha=0.9)
axes[0].axhline(CO2_BACKGROUND_PPM, ls='--', color='gray', lw=1.5,
                label=f'Global background ~{CO2_BACKGROUND_PPM} ppm')
axes[0].set_ylabel('CO₂ (ppm)')
axes[0].set_title('CO₂ concentration')
axes[0].legend(fontsize=9)

axes[1].plot(_t_rel_z, _vz['temp_C'], lw=0.6, color='tab:red', alpha=0.9)
axes[1].set_ylabel('Temperature (°C)')
axes[1].set_title('Temperature')

axes[2].plot(_t_rel_z, _vz['alt_baro'], lw=0.6, color='tab:green', alpha=0.9)
axes[2].set_ylabel('Baro. altitude (m ASL)')
axes[2].set_xlabel('Time in flight window (min)')
axes[2].set_title('Altitude from pressure (ISA formula)')

for ax in axes:
    ax.axvspan(_t_apo_z, _t_lnd_z, color='tab:orange', alpha=0.2, label='Drop phase' if ax is axes[0] else None)
    ax.axvline(_t_apo_z, color='tab:orange', lw=1.0, ls='--')
    ax.axvline(_t_lnd_z, color='tab:orange', lw=1.0, ls=':')

plt.tight_layout()
save_figure(fig, 'fig_vamos_science.png')
plt.show()

In [ ]:
# ── 5.3b CO₂ — altitude scatter and distribution ─────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('VAMOS — CO₂ analysis (warm-up zeros excluded)', fontweight='bold')

ax = axes[0]
ax.scatter(vamos_co2['co2_ppm'], vamos_co2['alt_baro'],
           s=2, alpha=0.25, color='tab:brown')
ax.axvline(CO2_BACKGROUND_PPM, ls='--', color='gray', lw=1.8,
           label=f'Atmospheric background ~{CO2_BACKGROUND_PPM} ppm')
ax.set_xlabel('CO₂ (ppm)')
ax.set_ylabel('Barometric altitude (m ASL)')
ax.set_title('CO₂ vs altitude')
ax.legend(fontsize=9)

ax = axes[1]
ax.hist(vamos_co2['co2_ppm'], bins=60, color='tab:brown', alpha=0.7, edgecolor='none')
ax.axvline(CO2_BACKGROUND_PPM, ls='--', color='gray', lw=1.8,
           label=f'Background {CO2_BACKGROUND_PPM} ppm')
med_co2 = vamos_co2['co2_ppm'].median()
ax.axvline(med_co2, ls='-', color='darkred', lw=2,
           label=f'Observed median: {med_co2:.0f} ppm')
ax.set_xlabel('CO₂ (ppm)')
ax.set_ylabel('Sample count')
ax.set_title('CO₂ distribution')
ax.legend(fontsize=9)

plt.tight_layout()
save_figure(fig, 'fig_co2_analysis.png')
plt.show()

excess = med_co2 - CO2_BACKGROUND_PPM
print(f'Median CO₂: {med_co2:.0f} ppm  |  Excess above background: +{excess:.0f} ppm ({excess/CO2_BACKGROUND_PPM*100:.0f}%)')
print(f'Max CO₂: {vamos_co2["co2_ppm"].max():.0f} ppm — likely local plume or measurement artefact near ground')

### 5.4 Wind Analysis — VAMOS

The VAMOS anemometer records 2D wind speed, direction, and X/Y Cartesian components at ~2 Hz.
A second IMU channel records linear acceleration, from which we compute the **vector acceleration
magnitude** as a surrogate for tumbling intensity (the on-board tumbling flag is unreliable).

**Phase-separated spectral analysis** is critical here. The full 100-min recording contains three
physically distinct regimes:

| Phase | Description | Expected spectral signature |
|-------|-------------|-----------------------------|
| **Aircraft / cruise** | CanSat inside the balloon/drone, pre-ejection | High broadband power; vibration from motor/engine |
| **Drop / parachute** | Free descent under parachute | Moderate power; possible pendulum peak |
| **Ground / post-landing** | Sensor on the ground | Low power; ambient turbulence only |

Mixing all three phases into a single spectrum would produce an uninterpretable result.
The phase boundaries are derived from the pressure-based `detect_vamos_drop()` function.

In [ ]:
# ── 5.4 Wind — time series and rose ──────────────────────────────────────────

drop_t0_h = (vamos_drop['t_apogee_s']  - wind['t_s'].iat[0]) / 3600
drop_t1_h = (vamos_drop['t_landing_s'] - wind['t_s'].iat[0]) / 3600
t_wh      = wind['t_rel'].values / 3600

fig = plt.figure(figsize=(14, 8))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.38, wspace=0.45)
fig.suptitle('VAMOS wind — time series and wind rose', fontweight='bold')

ax = fig.add_subplot(gs[0, :2])
ax.plot(t_wh, wind['wind_spd'], lw=0.5, color='steelblue', alpha=0.8)
ax.axvspan(drop_t0_h, drop_t1_h, color='tab:orange', alpha=0.18, label='Drop phase')
ax.set_ylabel('Wind speed (m s⁻¹)')
ax.set_xlabel('Time (h)')
ax.set_title('(a) Wind speed time series')
ax.legend(fontsize=8)

ax = fig.add_subplot(gs[1, :2])
ax.plot(t_wh, wind['x_mps'], lw=0.5, color='tab:blue',  alpha=0.7, label='X (E–W)')
ax.plot(t_wh, wind['y_mps'], lw=0.5, color='tab:green', alpha=0.7, label='Y (N–S)')
ax.axvspan(drop_t0_h, drop_t1_h, color='tab:orange', alpha=0.18)
ax.set_ylabel('Wind component (m s⁻¹)')
ax.set_xlabel('Time (h)')
ax.set_title('(b) Wind components with drop-phase highlight')
ax.legend(fontsize=8)

ax = fig.add_subplot(gs[:, 2], projection='polar')
theta = np.deg2rad(wind['wind_dir'].values)
r     = wind['wind_spd'].values
sc    = ax.scatter(theta, r, s=3, alpha=0.25, c=wind['t_rel'] / 60, cmap='YlOrRd')
ax.set_theta_zero_location('N')
ax.set_theta_direction(-1)
ax.set_title('(c) Wind rose\n(colour = mission minute)', pad=15)
plt.colorbar(sc, ax=ax, label='Minutes from wind start', pad=0.12, fraction=0.04)

save_figure(fig, 'fig_wind_overview.png')
plt.show()

print(f'Mean wind speed: {wind["wind_spd"].mean():.2f} m s⁻¹  |  Max: {wind["wind_spd"].max():.2f} m s⁻¹')
print(f'Drop duration (pressure-based): {vamos_drop["drop_duration_s"]:.0f} s  |  '
      f'Mean descent rate: {vamos_drop["descent_rate_mps"]:.2f} m s⁻¹')

In [ ]:
# ── 5.4b Wind — phase-aware spectral analysis ─────────────────────────────────

fs_psd    = 1.0
nperseg_p = 64

in_plane_idx = np.where(vamos['press_hPa'].values < vamos_drop['p0_hPa'] - 30)[0]
if len(in_plane_idx):
    t_plane_start = vamos['t_s'].iloc[in_plane_idx[0]] + 60
    t_plane_end   = vamos_drop['t_apogee_s'] - 30
else:
    t_plane_start = wind['t_s'].min()
    t_plane_end   = vamos_drop['t_apogee_s']

phase_masks = {
    'aircraft / cruise': ((wind['t_s'] > t_plane_start) & (wind['t_s'] < t_plane_end), 'tab:blue'),
    'drop / parachute':  ((wind['t_s'] >= vamos_drop['t_apogee_s']) &
                          (wind['t_s'] <= vamos_drop['t_landing_s']), 'tab:red'),
    'ground / post-landing': (wind['t_s'] > vamos_drop['t_landing_s'] + 30, '0.4'),
}

fs_spec      = max(1.0, round(FS_W))
tu_w, wacc_u = resample_uniform(wind['t_s'].values, wind['wind_acc_vec'].values, fs_spec)
wacc_u      -= wacc_u.mean()
nperseg_s    = int(min(256, max(64, len(wacc_u) // 8)))
f_wind, t_wind, Sww = spectrogram(
    wacc_u, fs=fs_spec, window='hann',
    nperseg=nperseg_s, noverlap=int(nperseg_s * 0.75), scaling='density')

fig = plt.figure(figsize=(14, 10))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.34, wspace=0.28, height_ratios=[1, 1.15])
fig.suptitle('VAMOS wind — phase-aware spectral analysis', fontweight='bold')

ax = fig.add_subplot(gs[0, 0])
ax.plot(vamos['t_rel'] / 60, vamos['press_hPa'], lw=0.8, color='tab:blue')
ax.axhline(vamos_drop['p0_hPa'], ls=':', color='0.4', lw=1.2,
           label=f'p₀ ≈ {vamos_drop["p0_hPa"]:.1f} hPa')
ax.axvspan((vamos_drop['t_apogee_s']  - vamos['t_s'].iat[0]) / 60,
           (vamos_drop['t_landing_s'] - vamos['t_s'].iat[0]) / 60,
           color='tab:orange', alpha=0.2, label='Drop phase')
ax.set_xlabel('Time from start (min)')
ax.set_ylabel('Pressure (hPa)')
ax.set_title('(a) Pressure — phase boundaries')
ax.legend(fontsize=8)

ax = fig.add_subplot(gs[1, 0])
t_spec_abs = (t_wind + tu_w[0] - wind['t_s'].iat[0]) / 60
pcm = ax.pcolormesh(t_spec_abs, f_wind,
                    10 * np.log10(Sww + 1e-12), cmap='viridis', shading='gouraud')
ax.axvline((vamos_drop['t_apogee_s']  - wind['t_s'].iat[0]) / 60,
           color='r', ls='--', lw=1, label='Apogee')
ax.axvline((vamos_drop['t_landing_s'] - wind['t_s'].iat[0]) / 60,
           color='g', ls='--', lw=1, label='Landing')
ax.set_xlabel('Time from wind start (min)')
ax.set_ylabel('Frequency (Hz)')
ax.set_ylim(0, fs_spec / 2)
ax.set_title(f'(b) STFT of |wind acceleration|  (fs = {fs_spec:.0f} Hz)')
ax.legend(fontsize=8)
fig.colorbar(pcm, ax=ax, label='dB re (m s⁻²)² Hz⁻¹')

ax = fig.add_subplot(gs[0, 1])
for name, (mask, color) in phase_masks.items():
    if mask.sum() < 32:
        continue
    f, P, Nu, dof = compute_welch_psd(
        wind.loc[mask, 't_s'], wind.loc[mask, 'wind_acc_vec'],
        fs=fs_psd, nperseg=nperseg_p)
    ax.loglog(f[1:], P[1:], color=color, lw=1.5,
              label=f'{name} (N={Nu}, DoF≈{dof:.0f})')
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('PSD ((m s⁻²)² Hz⁻¹)')
ax.set_title('(c) Welch PSD — vector acceleration')
ax.legend(fontsize=8, loc='lower left')

ax = fig.add_subplot(gs[1, 1])
for name, (mask, color) in phase_masks.items():
    if mask.sum() < 32:
        continue
    f, P, Nu, dof = compute_welch_psd(
        wind.loc[mask, 't_s'], wind.loc[mask, 'wind_spd'],
        fs=fs_psd, nperseg=nperseg_p)
    ax.loglog(f[1:], P[1:], color=color, lw=1.5,
              label=f'{name} (N={Nu}, DoF≈{dof:.0f})')
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('PSD ((m s⁻¹)² Hz⁻¹)')
ax.set_title('(d) Welch PSD — wind speed')
ax.legend(fontsize=8, loc='lower left')

plt.tight_layout()
save_figure(fig, 'fig_wind_spectrogram.png')
plt.show()

print(f'Peak VAMOS altitude: {vamos_drop["h_peak_m"]:.0f} m AGL')
print('Phase-separated PSD isolates aircraft, parachute and ground regimes — '
      'mixing them into one spectrum would blur physically distinct energy distributions.')

### 5.5 External Radiosonde — UWYO Station 06610 (Payerne, CH)

The University of Wyoming upper-air sounding for **station 06610 at 2026-02-05 12:00 UTC**
(Payerne, Switzerland, ~40 km SW of Dübendorf) provides synoptic context for the CanSat flight.

**What the Skew-T / log-P diagram shows:**
- **Temperature profile** (right solid curve): actual atmospheric temperature vs pressure.
- **Dew point profile** (left solid curve): gap between temperature and dew point indicates dryness.
- **Dry adiabats** (red dashed): parcel temperature if lifted dry-adiabatically — divergence from
  actual temperature indicates stability.
- **Moist adiabats** (blue dashed): parcel trajectory after condensation.
- **Mixing ratio lines** (green dashed): constant specific humidity.
- **Wind barbs**: direction and speed at each pressure level.

**Important caveat:** Payerne is a regional synoptic reference, not a co-located sounding.
Local orography and mesoscale effects mean that the exact thermodynamic profile above Dübendorf
may differ. This sounding should be interpreted as **regional atmospheric context**, not
ground-truth validation of the CanSat measurements.

In [ ]:
# ── 5.5 UWYO radiosonde — Skew-T / log-P + vertical wind shear ───────────────

uwyo_prof = uwyo.dropna(subset=['pressure_hPa', 'height_m', 'temperature_C',
                                 'dewpoint_C', 'wind_dir_deg', 'wind_spd_ms']).copy()
uwyo_prof = uwyo_prof.sort_values('height_m').reset_index(drop=True)
uwyo_prof['u_ms'], uwyo_prof['v_ms'] = met_to_uv(
    uwyo_prof['wind_dir_deg'], uwyo_prof['wind_spd_ms'])

surface_layer = uwyo_prof[uwyo_prof['height_m'] <= uwyo_prof['height_m'].min() + 50]
u0 = surface_layer['u_ms'].median()
v0 = surface_layer['v_ms'].median()
uwyo_prof['bulk_shear_ms'] = np.hypot(uwyo_prof['u_ms'] - u0, uwyo_prof['v_ms'] - v0)

uwyo_layer = (
    uwyo_prof.assign(layer_m=(250 * np.round(uwyo_prof['height_m'] / 250)).astype(int))
    .groupby('layer_m', as_index=False)
    .agg({'height_m': 'mean', 'pressure_hPa': 'mean',
          'u_ms': 'mean', 'v_ms': 'mean', 'bulk_shear_ms': 'mean'})
)
du = np.diff(uwyo_layer['u_ms'])
dv = np.diff(uwyo_layer['v_ms'])
dz = np.diff(uwyo_layer['height_m'])
layer_pressure = 0.5 * (uwyo_layer['pressure_hPa'].values[1:] + uwyo_layer['pressure_hPa'].values[:-1])
layer_shear    = np.where(dz > 0, np.hypot(du, dv) / (dz / 1000), np.nan)

mission_h = grasp_clean['alt_baro'].mean()
mission_p = np.interp(mission_h, uwyo_prof['height_m'], uwyo_prof['pressure_hPa'])
bulk_1km  = np.interp(uwyo_prof['height_m'].min() + 1000,
                      uwyo_prof['height_m'], uwyo_prof['bulk_shear_ms'])
barb_stride = max(1, len(uwyo_prof) // 38)

try:
    station_lat = uwyo_prof['latitude'].iloc[0]
    station_lon = uwyo_prof['longitude'].iloc[0]
    station_str = f'Latitude: {station_lat:.3f}  Longitude: {station_lon:.3f}'
except KeyError:
    station_str = 'Payerne, Switzerland'

fig = plt.figure(figsize=(13.8, 7.4))
gs  = gridspec.GridSpec(1, 2, figure=fig, width_ratios=[1.9, 0.9], wspace=0.18)
fig.subplots_adjust(top=0.86, left=0.07, right=0.98, bottom=0.08)
fig.text(0.07, 0.988, 'Station 06610 at 12 UTC 05 Feb 2026',
         ha='left', va='top', fontsize=17)
fig.text(0.07, 0.955, f'PAYERNE, Switzerland\n{station_str}',
         ha='left', va='top', fontsize=9)

ax0 = fig.add_subplot(gs[0, 0])
plot_uwyo_skewt(ax0, uwyo_prof.sort_values('pressure_hPa', ascending=False),
                barb_stride=barb_stride)

ax1 = fig.add_subplot(gs[0, 1], sharey=ax0)
ax1.plot(uwyo_prof['bulk_shear_ms'], uwyo_prof['pressure_hPa'],
         color='darkorange', lw=2.2, label='Bulk shear from surface')
ax1.fill_betweenx(uwyo_prof['pressure_hPa'], 0, uwyo_prof['bulk_shear_ms'],
                  color='orange', alpha=0.14)
ax1.plot(layer_shear, layer_pressure,
         color='crimson', lw=1.5, label='Layer shear (250 m bins)')
ax1.axhline(mission_p, color='0.25', ls='--', lw=1.2,
            label=f'GRASP mean alt ≈ {mission_h:.0f} m ASL')
ax1.set_xlabel('Shear (m s⁻¹ or m s⁻¹ km⁻¹)')
ax1.set_title('Vertical wind shear')
ax1.grid(True, alpha=0.25)
ax1.legend(fontsize=8, loc='lower right')
ax1.tick_params(labelleft=False)

saved_path = save_figure(fig, 'fig_uwyo_skewt_shear.png')
plt.show()

print(f'UWYO surface temperature: {uwyo_prof["temperature_C"].iloc[0]:.1f} °C  |  '
      f'dew point: {uwyo_prof["dewpoint_C"].iloc[0]:.1f} °C')
print(f'0–1 km bulk shear: {bulk_1km:.1f} m s⁻¹')
print(f'Max bulk shear in sounding: {uwyo_prof["bulk_shear_ms"].max():.1f} m s⁻¹')
print(f'Figures saved to: {FIG_DIR.resolve()}')
print(f'Original UWYO Skew-T URL: {UWYO_SKEWT_URL}')

---
## Section 6 — Discussion & Conclusions

### 6.1 Temperature and Lapse Rate

The concurrent VAMOS–GRASP two-point comparison yields an **observed lapse rate that deviates
significantly from the ISA standard (−6.5 K km⁻¹)**. A sub-standard lapse rate (temperature
decreasing less rapidly with altitude than ISA, or even increasing) indicates a **temperature
inversion**, consistent with a stable, warm air mass above the planetary boundary layer on a
winter morning.

GRASP temperatures are **~7 K above the ISA prediction** at the same barometric altitude —
a positive bias typical of a continental winter anticyclone with a strong surface inversion.
The OBAMA external dataset confirms ground temperatures consistent with VAMOS, further
validating the absolute temperature calibration of both sensor suites.

**Caveat:** with only ~40 m of barometric altitude range in GRASP and just two points in the
lapse-rate estimate (GRASP mean + VAMOS concurrent median), the magnitude of the derived
lapse rate carries substantial uncertainty. The sign (inversion) is, however, physically
coherent with the synoptic situation.

### 6.2 Particulate Matter

PM₂.₅ and PM₁₀ show a **clear increasing trend toward lower altitudes** over the narrow
GRASP altitude range, consistent with boundary-layer aerosol accumulation near the surface.
Approximately 15% of GRASP PM₂.₅ samples exceed the WHO 2021 24-h guideline
(15 µg m⁻³), indicating an elevated local aerosol load above the Dübendorf launch site.
Peak values near landing point to localised combustion or traffic influence.

### 6.3 CO₂

The median VAMOS CO₂ reading (~560 ppm) exceeds the global atmospheric background
(420 ppm) by **~33%**. Values up to ~1 300 ppm are observed; these spikes likely correspond
to proximity to traffic or combustion sources at or near the ground in the peri-urban
Dübendorf environment. The absolute accuracy of the NDIR sensor (±50 ppm without factory
temperature compensation) means trends are more reliable than absolute values.

### 6.4 Signal Processing

The **phase-separated Welch PSD** and STFT spectrogram of wind acceleration reveal that
spectral energy is strongly phase-dependent: the aircraft/cruise phase shows the highest
broadband power, the parachute drop phase shows intermediate power, and the ground phase
shows the lowest. **No sharp spectral peak indicative of a parachute pendulum resonance**
is detected in either the pressure or wind spectra, suggesting a stable, non-oscillatory descent.

### 6.5 External Sounding

The UWYO station 06610 sounding provides regional thermodynamic context consistent with a
stable lower atmosphere on the morning of 5 February 2026. The wind shear profile shows
increasing bulk shear aloft, which aligns with the wind direction variability observed by VAMOS
during the descent.

---

## Limitations

| Limitation | Impact |
|-----------|--------|
| Narrow GRASP altitude range (~40 m barometrically) | Lapse rate is a two-point estimate with large uncertainty |
| Temporal offset of OBAMA vs GRASP (not guaranteed simultaneous) | Temperature comparison is approximate |
| GPS altitude noise (σ ≈ 10–20 m) | GPS vertical profiling unusable; barometric preferred |
| NDIR CO₂ sensor without T compensation | Absolute CO₂ accuracy limited to ~±50 ppm |
| Tumbling flag stuck at 0 | Cannot assess payload rotation from on-board algorithm |
| UWYO sounding not co-located (Payerne, ~40 km SW) | Regional context only, not local ground-truth |

---

## Conclusions

1. The lower troposphere above Dübendorf on 5 February 2026 was **~7 K warmer than ISA**,
   with a sub-standard temperature lapse rate indicative of a **stable winter boundary layer**
   and/or surface temperature inversion.

2. **PM₂.₅ and PM₁₀ increased towards the surface**, with ~15% of GRASP PM₂.₅ samples
   exceeding the WHO 24-h guideline, reflecting boundary-layer aerosol accumulation at the
   peri-urban Dübendorf site.

3. **CO₂ was ~33% above the global background** at ground level, driven by local traffic
   and combustion sources — a characteristic urban fingerprint.

4. **Phase-separated spectral analysis** revealed no parachute pendulum resonance and showed
   that spectral energy distribution differs markedly between aircraft cruise, parachute drop,
   and ground phases — confirming the value of phase-aware analysis.

5. Three instrument artefacts were identified and corrected before analysis: a GRASP
   sensor-init pressure spike (row 0), a VAMOS CO₂ warm-up period (~7 min), and an
   always-zero tumbling flag.

6. The **UWYO radiosonde** provides regional synoptic context consistent with the CanSat
   in-situ observations, supporting interpretation of a stable, sheared lower atmosphere.